# Steady-state values of PVs

In [2]:
import numpy as np
from scipy.optimize import least_squares as lsq

pi = np.pi
sqrt = np.sqrt
exp = np.exp

### Column dimensions ###
d = 1 # m, diameter
r = 0.5*d # m, radius
h = 10*d # m, height
A_cs = (pi/4)*d**2 # m², cross-sectional area
h_outlet = 9.5 # m, outlet @ 95% column height
V_t = A_cs*h_outlet # m³, column volume
A_h = 2*pi*r*h_outlet # m², column surface area over which heat loss occurs

### Properties ###
H = 280 # J.gₐᵤ⁻¹, ΔH from literature, converted from kJ.mol⁻¹ to J.gₐᵤ⁻¹
E_A = 20310*4.184 # J.mol⁻¹, from literature
k_d0 = 0.7/60 # min⁻¹ @ 148 °C, from literature
T_0 = 421.15 # K, from literature
R = 8.314 # J.mol⁻¹.K⁻¹
T_new = 403.15 # K (130 °C)
k_d = k_d0*exp((-E_A/R)*(1/T_new - 1/T_0)) # min⁻¹, desorption rate constant
G = -2022*4.184 # J.mol⁻¹, Gibbs free energy, ΔG of adsorption 
epsilon = 0.292 # void fraction
delta = 1 - epsilon
gamma = delta/epsilon
rho_C = 838 # kg.m⁻³, carbon
rho_E = 934.62 # kg.m⁻³, eluant
rho_O = 986 # kg.m⁻³, thermal oil
c_pC = 925 # J.kg⁻¹.K⁻¹, carbon
c_pE = 4263 # J.kg⁻¹.K⁻¹, eluant
c_pO = 2643 # J.kg⁻¹.K⁻¹, thermal oil
C_pE = rho_E*c_pE # J·K⁻¹·m⁻³, volumetric heat capacity
C_pC = rho_C*c_pC # same
C_pO = rho_O*c_pO # same
m_st = rho_E*epsilon*V_t # kg, mass of eluant in column
m_ct = rho_C*delta*V_t # kg꜀, mass of carbon in column
T_E0 = 403.15 # K (130 °C), inlet eluant temperature, from HX, ideally
T_C0 = 383.15 # K (110 °C), inlet carbon temperature, from acid wash (AW)
# Thermal oil = Xceltherm 600 (diphenyl oxide-biphenyl mixture) (Alva et al., 2018:350)
T_1 = 473.15 # K (200 °C), thermal oil feed to HX
T_2 = 318.15 # K (45 °C), thermal oil @ HX outlet
t_res = 14 # h, residence time
t_1 = 298.15 # K (25 °C), softened water feed to HX, @ ambient temperature
t_2 = T_E0 # K (130 °C), softened water (eluant) @ HX outlet/inlet to column
T_amb = t_1 # K, ambient air temperature surrounding column
K_b = 2.06e9 # Pa = kg·m⁻¹·min⁻²
# Linearly interpolate for β (between 120 °C and 140 °C) to find @ 130 °C (from EngineeringToolbox)
beta = (130 - 120)/(140 - 120)*(9.75e4 - 8.6e4) + 8.6e4 # K⁻¹
A = 1/((C_pE*epsilon + C_pC*delta)*V_t) # K·J⁻¹

### Selected parameters ###
F_C_ss = delta*V_t/(t_res*60) # m³.min⁻¹, volumetric flow rate of carbon into column
BV = delta*V_t # m³, bed volume
num_BV = 2
Flow_eluant = num_BV*BV/1 # m³.h⁻¹, 5 BV.h⁻¹ flow rate FOR NOW
F_E_ss = Flow_eluant/60 # m³.min⁻¹, volumetric flow rate of eluant into column

# HX duty 
rho_E_avg = 965.84 # kg.m⁻³
c_pE_avg = 4.221 # kJ.kg⁻¹.K⁻¹
m_E = rho_E_avg*F_E_ss # kg.min⁻¹, mass flow rate of eluant (cold fluid)
q_c = m_E*c_pE_avg*(t_2 - t_1) # kJ.min⁻¹, heat duty cold-side
# Assume q_loss = 0 @ HX
# q_h = q_c
Q_heater = q_c/60 # kJ.s⁻¹ (1256 kW heat duty)
q_h = q_c 
m_O = q_h/((c_pO/1000)*(T_1 - T_2)) # kg.min⁻¹, mass flow rate of oil (hot fluid)
F_O_ss = m_O/rho_O # m³.min⁻¹, volumetric flow rate of oil

# Concentrations
C_s0 = 0 # no gold in solution initially
q_0 = 2.5 # gₐᵤ.kg꜀⁻¹, initial gold loading on carbon (2500 g.ton⁻¹)

# BPV opening
kappa = 0.6
# Downstream pressure = atmospheric (on EW side)
P_d = 0 # Pa = kg·m⁻¹·min⁻² (gauge pressure)
# Valve coefficient
C_V = 0.023 # m²
# Overall heat transfer coefficient (for heat loss from column)
U = 100*60 # J·min⁻¹·K⁻¹·m⁻²

### Solve for steady-state values ###

def functions(variables):
    C_s, q, T, P = variables

    # Solve for Cₛ, q, T, P
    # Define functions from RHS of ODEs
    # From dCₛ/dt = f_1
    # From dq/dt = f_2
    # From dT/dt = f_3
    # From dP/dt = f_4

    f_1 = 1/(epsilon*V_t)*(F_E_ss*C_s0 - kappa*C_V*C_s*sqrt((P - P_d)/rho_E)) + gamma*rho_C*k_d0*exp((-E_A/R)*(1/T - 1/T_0))*(q - C_s*exp(-G/(R*T)))
    f_2 = F_C_ss/(delta*V_t)*(q_0 - q) - k_d0*exp((-E_A/R)*(1/T - 1/T_0))*(q - C_s*exp(-G/(R*T)))

    # f_3 & f_4 enclosed in brackets to write long expressions over two lines
    f_3 = (C_pE*A*(F_E_ss*T_E0 - kappa*C_V*T*sqrt((P - P_d)/rho_E)) + C_pC*F_C_ss*A*(T - T_C0) + C_pO*F_O_ss*A*(T_1 - T_2) - U*A_h*A*(T - T_amb) 
        - delta*rho_C*k_d0*V_t*H*A*exp((-E_A/R)*(1/T - 1/T_0))*(q - C_s*exp(-G/(R*T))))
    f_4 = (K_b/(epsilon*V_t)*(F_E_ss - kappa*C_V*sqrt((P - P_d)/rho_E)) + C_pE*A*beta*K_b*(F_E_ss*T_E0 - kappa*C_V*T*sqrt((P - P_d)/rho_E)) 
        + C_pC*F_C_ss*A*beta*K_b*(T - T_C0) + C_pO*F_O_ss*A*beta*K_b*(T_1 - T_2) - U*A_h*A*beta*K_b*(T - T_amb) - delta*rho_C*k_d0*V_t*H*A*beta*K_b*exp((-E_A/R)*(1/T - 1/T_0))*(q - C_s*exp(-G/(R*T))))

    # Define residuals

    residual_1 = f_1
    residual_2 = f_2
    residual_3 = f_3
    residual_4 = f_4
    
    return [residual_1, residual_2, residual_3, residual_4]

# Initial estimates
# Cₛ = 100 gₐᵤ.m⁻³
# q = 1 gₐᵤ.kg꜀⁻¹
# T = 373.15 K
# P = 380000 Pa = 380 kPa

estimates = [100, 1, 373.15, 280000]

# Define lower bounds
lower = [0.01, 0.01, 298.15, 101330]

# Define upper bounds
upper = [500, 2.5, 403.15, 500000]

bounds = (lower, upper)

solutions = lsq(functions, x0=estimates, bounds=bounds)

C_s_ss = solutions.x[0]
q_ss = solutions.x[1]
T_ss = solutions.x[2]
P_ss = solutions.x[3]

In [3]:
C_V = 2*F_E_ss/sqrt(220000/rho_E)
print(f'C_V = {C_V:.3f}')

C_V = 0.023


In [4]:
print(f'{C_s_ss:.2f} gₐᵤ.m⁻³')

119.17 gₐᵤ.m⁻³


In [5]:
print(f'{q_ss:.2f} gₐᵤ.kg꜀⁻¹')

0.95 gₐᵤ.kg꜀⁻¹


In [6]:
print(f'{T_ss:.2f} K')

369.55 K


In [7]:
print(f'{P_ss:.2f} Pa or {P_ss/1000:.2f} kPa (gauge)')

271435.68 Pa or 271.44 kPa (gauge)


# Values for matrix coefficients

In [8]:
### Matrix coefficients ###

a_11 = -C_V*kappa*sqrt((P_ss - P_d)/rho_E)/(V_t*epsilon) - gamma*k_d0*rho_C*exp(E_A/(R*T_0) - E_A/(R*T_ss) - G/(R*T_ss))
a_12 = gamma*k_d0*rho_C*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0))
a_13 = (gamma*k_d0*rho_C*(-C_s_ss*G - E_A*(C_s_ss - q_ss*exp(G/(R*T_ss))))*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0) - G/(R*T_ss)))/(R*T_ss**2)
a_14 = -C_V*C_s_ss*kappa*sqrt((P_ss - P_d)/rho_E)/(2*V_t*epsilon*(P_ss - P_d))
a_21 = k_d0*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0) - G/(R*T_ss))
a_22 = -F_C_ss/(V_t*delta) - k_d0*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0))
a_23 = (k_d0*(C_s_ss*G + E_A*(C_s_ss - q_ss*exp(G/(R*T_ss))))*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0) - G/(R*T_ss)))/(R*T_ss**2)
a_24 = 0
a_31 = A*H*V_t*delta*k_d0*rho_C*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0) - G/(R*T_ss))
a_32 = -A*H*V_t*delta*k_d0*rho_C*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0))
a_33 = (1/(R*T_ss**2))*(A*(C_s_ss*G*H*V_t*delta*k_d0*rho_C*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0)) + 
                           E_A*H*V_t*delta*k_d0*rho_C*(C_s_ss - q_ss*exp(G/(R*T_ss)))*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0)) +
                           R*T_ss**2*(-A_h*U - C_V*C_pE*kappa*sqrt((P_ss - P_d)/rho_E) + C_pC*F_C_ss)*exp(G/(R*T_ss)))*exp(-G/(R*T_ss)))
a_34 = -A*C_V*C_pE*T_ss*kappa*sqrt((P_ss - P_d)/rho_E)/(2*(P_ss - P_d))
a_41 = A*H*K_b*V_t*beta*delta*k_d0*rho_C*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0) - G/(R*T_ss))
a_42 = -A*H*K_b*V_t*beta*delta*k_d0*rho_C*exp(E_A*(T_ss - T_0)/(R*T_ss*T_0))
a_43 = (-A*A_h*K_b*U*beta - A*C_V*C_pE*K_b*beta*kappa*sqrt((P_ss - P_d)/rho_E) + A*C_pC*F_C_ss*K_b*beta +
        A*C_s_ss*E_A*H*K_b*V_t*beta*delta*k_d0*rho_C*exp(E_A/(R*T_0) - E_A/(R*T_ss) - G/(R*T_ss))/(R*T_ss**2) +
        A*C_s_ss*G*H*K_b*V_t*beta*delta*k_d0*rho_C*exp(E_A/(R*T_0) - E_A/(R*T_ss) - G/(R*T_ss))/(R*T_ss**2) -
        A*E_A*H*K_b*V_t*beta*delta*k_d0*q_ss*rho_C*exp(E_A/(R*T_0) - E_A/(R*T_ss))/(R*T_ss**2))
a_44 = C_V*K_b*kappa*sqrt((P_ss - P_d)/rho_E)*(-A*C_pE*T_ss*V_t*beta*epsilon - 1)/(2*V_t*epsilon*(P_ss - P_d))

b_11 = C_s0/(V_t*epsilon)
b_12 = 0; b_13 = 0
b_14 = -C_V*C_s_ss*kappa*sqrt((P_ss - P_d)/rho_E)/(V_t*epsilon)
b_21 = 0; b_23 = 0; b_24 = 0
b_22 = (-q_ss + q_0)/(V_t*delta)
b_31 = A*C_pE*T_E0
b_32 = A*C_pC*(T_ss - T_C0)
b_33 = A*C_pO*(T_1 - T_2)
b_34 = -A*C_V*C_pE*T_ss*sqrt((P_ss - P_d)/rho_E)
b_41 = A*C_pE*K_b*T_E0*beta + K_b/(V_t*epsilon)
b_42 = A*C_pC*K_b*beta*(T_ss - T_C0)
b_43 = A*C_pO*K_b*beta*(T_1 - T_2)
b_44 = C_V*K_b*sqrt((P_ss - P_d)/rho_E)*(-A*C_pE*T_ss*V_t*beta*epsilon - 1)/(V_t*epsilon)

e_11 = F_E_ss/(V_t*epsilon)
e_12 = 0; e_13 = 0; e_14 = 0; e_15 = 0
e_21 = 0; e_23 = 0; e_24 = 0; e_25 = 0
e_22 = F_C_ss/(V_t*delta)
e_31 = 0; e_32 = 0; 
e_33 = A*C_pE*F_E_ss
e_34 = -A*C_pC*F_C_ss
e_35 = A*C_pO*F_O_ss
e_41 = 0; e_42 = 0
e_43 = A*C_pE*F_E_ss*K_b*beta
e_44 = -A*C_pC*F_C_ss*K_b*beta
e_45 = A*C_pO*F_O_ss*K_b*beta

# A matrix
print("A matrix:")
print(f"a₁₁: {a_11:.8f}")
print(f"a₁₂: {a_12:.8f}")
print(f"a₁₃: {a_13:.8f}")
print(f"a₁₄: {a_14:.8f}")
print(f"a₂₁: {a_21:.8f}")
print(f"a₂₂: {a_22:.8f}")
print(f"a₂₃: {a_23:.8f}")
print(f"a₂₄: {a_24}")
print(f"a₃₁: {a_31:.8f}")
print(f"a₃₂: {a_32:.8f}")
print(f"a₃₃: {a_33:.8f}")
print(f"a₃₄: {a_34:.8f}")
print(f"a₄₁: {a_41:.8f}")
print(f"a₄₂: {a_42:.8f}")
print(f"a₄₃: {a_43:.8f}")
print(f"a₄₄: {a_44:.8f} \n\n")
                 
# B matrix
print("B matrix:")
print(f"b₁₁: {b_11}")
print(f"b₁₂: {b_12}")
print(f"b₁₃: {b_13}")
print(f"b₁₄: {b_14:.8f}")
print(f"b₂₁: {b_21}")
print(f"b₂₂: {b_22:.8f}")
print(f"b₂₃: {b_23}")
print(f"b₂₄: {b_24}")
print(f"b₃₁: {b_31:.8f}")
print(f"b₃₂: {b_32:.8f}")
print(f"b₃₃: {b_33:.8f}")
print(f"b₃₄: {b_34:.8f}")
print(f"b₄₁: {b_41:.8f}")
print(f"b₄₂: {b_42:.8f}")
print(f"b₄₃: {b_43:.8f}")
print(f"b₄₄: {b_44:.8f} \n\n")                 

# E matrix
print("E matrix:")
print(f"e₁₁: {e_11:.8f}")
print(f"e₁₂: {e_12}")
print(f"e₁₃: {e_13}")
print(f"e₁₄: {e_14}")
print(f"e₁₅: {e_15}")
print(f"e₂₁: {e_21}")
print(f"e₂₂: {e_22:.8f}")
print(f"e₂₃: {e_23}")
print(f"e₂₄: {e_24}")
print(f"e₂₅: {e_25}")
print(f"e₃₁: {e_31}")
print(f"e₃₂: {e_32}")
print(f"e₃₃: {e_33:.8f}")
print(f"e₃₄: {e_34:.8f}")
print(f"e₃₅: {e_35:.8f}")
print(f"e₄₁: {e_41}")
print(f"e₄₂: {e_42}")
print(f"e₄₃: {e_43:.8f}")
print(f"e₄₄: {e_44:.8f}")
print(f"e₄₅: {e_45:.8f} \n\n\n")

### Time constants & gains ###
tau_p1 = 1/(-a_11)
tau_p2 = 1/(-a_22)
tau_p3 = 1/(-a_33)
tau_p4 = 1/(-a_44)

K_p1 = b_11/(-a_11)
K_d11 = a_12/(-a_11)
K_d12 = a_13/(-a_11)
K_d13 = a_14/(-a_11)
K_d14 = b_14/(-a_11)
K_d15 = e_11/(-a_11)

K_p2 = b_22/(-a_22)
K_d21 = a_21/(-a_22)
K_d22 = a_23/(-a_22)
K_d23 = e_22/(-a_22)

K_p3 = b_33/(-a_33)
K_d31 = a_31/(-a_33)
K_d32 = a_32/(-a_33)
K_d33 = a_34/(-a_33)
K_d34 = b_31/(-a_33)
K_d35 = b_32/(-a_33)
K_d36 = b_34/(-a_33)
K_d37 = e_33/(-a_33)
K_d38 = e_34/(-a_33)
K_d39 = e_35/(-a_33)

K_p4 = b_44/(-a_44)
K_d41 = a_41/(-a_44)
K_d42 = a_42/(-a_44)
K_d43 = a_43/(-a_44)
K_d44 = b_41/(-a_44)
K_d45 = b_42/(-a_44)
K_d46 = b_43/(-a_44)
K_d47 = e_43/(-a_44)
K_d48 = e_44/(-a_44)
K_d49 = e_45/(-a_44)

print("Concentration loop: \n")
print(f"tauₚ₁: {tau_p1:.2f}")
print(f"Kₚ₁: {K_p1:.8f}")
print(f"Kd₁₁: {K_d11:.8f}")
print(f"Kd₁₂: {K_d12:.8f}") 
print(f"Kd₁₃: {K_d13:.8f}")
print(f"Kd₁₄: {K_d14:.8f}")
print(f"Kd₁₅: {K_d15:.8f} \n\n") 

print("Gold loading loop: \n")
print(f"tauₚ₂: {tau_p2:.2f}")
print(f"Kₚ₂: {K_p2:.6f}")
print(f"Kd₂₁: {K_d21:.8f}")
print(f"Kd₂₂: {K_d22:.8f}")
print(f"Kd₂₃: {K_d23:.8f} \n\n")

print("Temperature loop: \n")
print(f"tauₚ₃: {tau_p3:.2f}")
print(f"Kₚ₃: {K_p3:.8f}")
print(f"Kd₃₁: {K_d31:.8f}")
print(f"Kd₃₂: {K_d32:.8f}")
print(f"Kd₃₃: {K_d33:.8f}")
print(f"Kd₃₄: {K_d34:.8f}")
print(f"Kd₃₅: {K_d35:.8f}")
print(f"Kd₃₆: {K_d36:.8f}")
print(f"Kd₃₇: {K_d37:.8f}")
print(f"Kd₃₈: {K_d38:.8f}")
print(f"Kd₃₉: {K_d39:.8f} \n\n")

print("Pressure loop: \n")
print(f"tauₚ₄: {tau_p4:.10f}")
print(f"Kₚ₄: {K_p4:.8f}")
print(f"Kd₄₁: {K_d41:.8f}")
print(f"Kd₄₂: {K_d42:.8f}")
print(f"Kd₄₃: {K_d43:.8f}")
print(f"Kd₄₄: {K_d44:.8f}")
print(f"Kd₄₅: {K_d45:.8f}")
print(f"Kd₄₆: {K_d46:.8f}")
print(f"Kd₄₇: {K_d47:.8f}")
print(f"Kd₄₈: {K_d48:.8f}")
print(f"Kd₄₉: {K_d49:.8f}")

A matrix:
a₁₁: -12.67212424
a₁₂: 0.80041529
a₁₃: -100.84754754
a₁₄: -0.00002365
a₂₁: 0.00618368
a₂₂: -0.00158441
a₂₃: 0.04963304
a₂₄: 0
a₃₁: 0.00059996
a₃₂: -0.00003822
a₃₃: -0.08201906
a₃₄: -0.00004983
a₄₁: 113395769112.83946228
a₄₂: -7223881915.43015289
a₄₃: -15502012338397.25390625
a₄₄: -9418078179.75995636 


B matrix:
b₁₁: 0.0
b₁₂: 0
b₁₃: 0
b₁₄: -12.83843734
b₂₁: 0
b₂₂: 0.29293489
b₂₃: 0
b₂₄: 0
b₃₁: 125.73166443
b₃₂: -0.82488512
b₃₃: 31.61792565
b₃₄: -45.08526898
b₄₁: 23763914180252596.00000000
b₄₂: -155907411319177.28125000
b₄₃: 5975946038143787.00000000
b₄₄: -8521341634289965.00000000 


E matrix:
e₁₁: 0.08082192
e₁₂: 0
e₁₃: 0
e₁₄: 0
e₁₅: 0
e₂₁: 0
e₂₂: 0.00119048
e₂₃: 0
e₂₄: 0
e₂₅: 0
e₃₁: 0
e₃₂: 0
e₃₃: 0.05491658
e₃₄: -0.00038158
e₃₅: 0.03806547
e₄₁: 0
e₄₂: 0
e₄₃: 10379508371905.29882812
e₄₄: -72119731406.56036377
e₄₅: 7194564679514.59863281 



Concentration loop: 

tauₚ₁: 0.08
Kₚ₁: 0.00000000
Kd₁₁: 0.06316347
Kd₁₂: -7.95821960
Kd₁₃: -0.00000187
Kd₁₄: -1.01312433
Kd₁₅: 0.006377